## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Human Activity Recognition with a CNN

This notebook shows how to train and evaluate a 1D convolutional neural network (CNN) on the UCI HAR dataset. You used the same dataset in Assignment 8, with 561 features that the authors computed. Here, the CNN receives the **raw sensor signals** and learns its own features.

>Anguita, D., Ghio, A., Oneto, L., Parra, X., & Reyes-Ortiz, J. L. (2013). A public domain dataset for human activity recognition using smartphones. In *ESANN* (Vol. 3, pp. 437-442).

Thirty volunteers did six activities:
1. Walking
2. Walking upstairs
3. Walking downstairs
4. Sitting
5. Standing
6. Laying

The volunteers wore a smartphone on the waist. The phone recorded the accelerometer and the gyroscope at 50 samples/s. The authors cut the signals into **windows** of 2.56 s (128 samples). Consecutive windows overlap by 50%.

Each window has 9 channels:
- total acceleration: x, y, z (includes gravity)
- body acceleration: x, y, z (total acceleration without the gravity component)
- body gyroscope: x, y, z

The authors split the data **by subject**: 21 subjects (7,352 windows) in the training set and 9 subjects (2,947 windows) in the test set.

<div style="text-align:center">
<img src="images/SampleTask.png" alt="plot" width="800" />
</div>

Website: <https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones>

## Learning goals
- understand the tensor shapes for a 1D CNN: (windows, channels, length)
- scale each channel with statistics from the training data only
- keep validation subjects apart from the training subjects
- train a CNN, and choose the best epoch with the validation set (early stopping)
- evaluate the final model **one time** on the test set: confusion matrix, per-class metrics and ROC AUC

Many parts of this notebook are PyTorch code. Focus on the main steps, not on the details of the code.

**Run time:** about 1 minute on a CPU. In Colab, you can use a GPU (Runtime > Change runtime type), but it is not necessary.

In [ ]:
# PyTorch is already installed in Colab. On your own computer: pip install torch
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'   # for Mac computers with Apple silicon (must be set before importing torch)

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

## 1. Load the data

The raw signals are in `data/UCI_HAR_Dataset/train/Inertial Signals` and `.../test/Inertial Signals`. There is one text file for each channel. Each row of a file is one window of 128 samples.

We put the 9 channels together in one array with the shape **(windows, channels, length)**. This is the shape that PyTorch needs for `Conv1d`. We also load the activity labels and the subject of each window.

In [ ]:
data_path = 'data/UCI_HAR_Dataset'
channel_names = ['total_acc_x', 'total_acc_y', 'total_acc_z',
                 'body_acc_x',  'body_acc_y',  'body_acc_z',
                 'body_gyro_x', 'body_gyro_y', 'body_gyro_z']
class_names = ['Walking', 'Upstairs', 'Downstairs', 'Sitting', 'Standing', 'Laying']

def load_group(group):
    """Return X (windows, 9, 128), y (activity 0-5) and subject for 'train' or 'test'."""
    X = np.stack([np.loadtxt(f'{data_path}/{group}/Inertial Signals/{ch}_{group}.txt') for ch in channel_names], axis=1)
    y = np.loadtxt(f'{data_path}/{group}/y_{group}.txt').astype(int) - 1     # labels 1-6 -> 0-5
    subject = np.loadtxt(f'{data_path}/{group}/subject_{group}.txt').astype(int)
    return X.astype(np.float32), y, subject

X_train_all, y_train_all, subj_train_all = load_group('train')   # takes about 10 s
X_test, y_test, subj_test = load_group('test')

print('Train:', X_train_all.shape, '| Test:', X_test.shape)
print('Train subjects:', np.unique(subj_train_all))
print('Test subjects: ', np.unique(subj_test))

The training set has the shape `(7352, 9, 128)`: 7,352 windows, 9 channels, and 128 samples in each channel.

Let us plot the 9 channels of one window.

In [ ]:
idx = 4658
print(f'Class: {class_names[y_train_all[idx]]} | Subject: {subj_train_all[idx]}')

t = np.arange(128) / 50     # time in seconds (50 samples/s)
fig, axes = plt.subplots(3, 3, figsize=(15, 9), sharex=True)
for j, name in enumerate(channel_names):
    axes[j // 3, j % 3].plot(t, X_train_all[idx, j])
    axes[j // 3, j % 3].set_title(name)
for a in axes[-1]:
    a.set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 2. Validation subjects and scaling

We must choose the number of training epochs. If we choose it with the test set, the test result is too optimistic (this is data leakage, lecture 04.04). So we take **4 of the 21 training subjects** as a **validation set**. The test set stays apart until the end.

The channels have different units (g for acceleration, rad/s for the gyroscope). As in 05.01, the network trains better if the inputs are scaled. We scale each channel to mean 0 and standard deviation 1, with the mean and the standard deviation of the **training subjects only**.

In [ ]:
rng = np.random.RandomState(0)
val_subjects = rng.choice(np.unique(subj_train_all), size=4, replace=False)
is_val = np.isin(subj_train_all, val_subjects)

X_tr, y_tr = X_train_all[~is_val], y_train_all[~is_val]
X_val, y_val = X_train_all[is_val], y_train_all[is_val]
print('Validation subjects:', np.sort(val_subjects))
print(f'Train: {len(y_tr)} windows from {len(np.unique(subj_train_all[~is_val]))} subjects | Validation: {len(y_val)} windows from 4 subjects')

# one mean and one standard deviation per channel, from the training windows only
ch_mean = X_tr.mean(axis=(0, 2), keepdims=True)
ch_std = X_tr.std(axis=(0, 2), keepdims=True)
X_tr_s = (X_tr - ch_mean) / ch_std
X_val_s = (X_val - ch_mean) / ch_std
X_test_s = (X_test - ch_mean) / ch_std
print('Channel means (train):', ch_mean.ravel().round(3))

## 3. Send the data to PyTorch

There is too much data to send all of it to the model at the same time. Instead, we send small groups of windows, called **batches**. The model computes the gradient with one batch and changes its parameters. Then it uses the next batch. One **epoch** is one pass through all the batches.

During training, the batches are chosen at random from the training set (`shuffle=True`).

PyTorch uses two objects for this:
- a **Dataset** tells PyTorch where the data is and how to get one window
- a **DataLoader** makes the batches

In [ ]:
class HARDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)   # (windows, channels, length)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(HARDataset(X_tr_s, y_tr), batch_size=128, shuffle=True)
val_loader = DataLoader(HARDataset(X_val_s, y_val), batch_size=512, shuffle=False)
test_loader = DataLoader(HARDataset(X_test_s, y_test), batch_size=512, shuffle=False)

xb, yb = next(iter(train_loader))
print('One batch:', xb.shape, yb.shape)

## 4. Build the CNN

The CNN has two convolution blocks. Each block has:
- a convolution layer (32 kernels in block 1, 64 kernels in block 2, each kernel with 3 samples)
- a ReLU activation
- **dropout**: during training, it sets 50% of the values to 0 at random. This makes overfitting less likely.
- **max pooling**: it keeps the maximum of each pair of samples, so the signal becomes 2 times shorter

Then two fully connected layers change the embedding into 6 class scores.

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_channels=9, n_outputs=6, n_timesteps=128, kernel_size=3):
        super().__init__()
        self.features = nn.Sequential(
            # block 1
            nn.Conv1d(in_channels=n_channels, out_channels=32, kernel_size=kernel_size),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.MaxPool1d(kernel_size=2),
            # block 2
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=kernel_size),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.MaxPool1d(kernel_size=2),
        )
        with torch.no_grad():   # find the length of the embedding
            flat_dim = self.features(torch.zeros(1, n_channels, n_timesteps)).numel()
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 100),
            nn.ReLU(),
            nn.Linear(100, n_outputs),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN1D()
print(model)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

The model has about 200,000 parameters. For comparison, the logistic regression of Assignment 8 had 561 × 6 + 6 = 3,372 parameters.

## 5. Train with early stopping

We use two helper functions:
- `train_one_epoch`: one pass through the training batches, with gradient descent
- `evaluate_model`: the loss and the accuracy on a dataset, with no change to the model

After each epoch, we compute the **validation loss**. We keep a copy of the model at the epoch with the lowest validation loss. This is called **early stopping**.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()                       # training mode: dropout is on
    running_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += len(yb)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()                        # evaluation mode: dropout is off
    running_loss, total = 0.0, 0
    y_true, y_prob = [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        running_loss += criterion(logits, yb).item() * len(yb)
        total += len(yb)
        y_true.append(yb.cpu().numpy())
        y_prob.append(torch.softmax(logits, dim=1).cpu().numpy())
    y_true, y_prob = np.concatenate(y_true), np.concatenate(y_prob)
    y_pred = y_prob.argmax(axis=1)
    return running_loss / total, (y_pred == y_true).mean(), y_true, y_pred, y_prob

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

torch.manual_seed(0)
model = CNN1D().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20
history = []
best_val_loss, best_state, best_epoch = np.inf, None, 0

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _, _ = evaluate_model(model, val_loader, criterion, device)
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    if val_loss < best_val_loss:        # early stopping: keep the best model
        best_val_loss, best_epoch = val_loss, epoch + 1
        best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch+1:02d}/{epochs} | Train loss {train_loss:.3f}, acc {train_acc:.3f} | "
          f"Validation loss {val_loss:.3f}, acc {val_acc:.3f}")

model.load_state_dict(best_state)
print(f'\nBest epoch (lowest validation loss): {best_epoch}')

In [ ]:
history_df = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='Train')
ax[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='Validation')
ax[0].axvline(best_epoch, color='gray', linestyle='--', label='Best epoch')
ax[0].set_ylabel('Loss')
ax[1].plot(history_df['epoch'], history_df['train_acc'], marker='o', label='Train')
ax[1].plot(history_df['epoch'], history_df['val_acc'], marker='o', label='Validation')
ax[1].axvline(best_epoch, color='gray', linestyle='--', label='Best epoch')
ax[1].set_ylabel('Accuracy')
for a in ax:
    a.set_xlabel('Epoch')
    a.legend()
    a.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Two things to see in these plots:
- The training accuracy is lower than the validation accuracy. This is normal here: dropout is on during training, and off during validation.
- If the best epoch is the last epoch, the validation loss is still going down. You can train for more epochs.

## 6. Evaluate on the test set (one time)

Now we use the test set, for the first and only time.

In [ ]:
test_loss, test_acc, y_true, y_pred, y_prob = evaluate_model(model, test_loader, criterion, device)
val_acc_best = history_df.loc[best_epoch - 1, 'val_acc']
print(f'Validation accuracy (best epoch): {val_acc_best:.3f}')
print(f'Test accuracy:                    {test_acc:.3f}')
print(f'Test ROC AUC (one-vs-rest, mean of the 6 classes): {roc_auc_score(y_true, y_prob, multi_class="ovr"):.3f}')

ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names, xticks_rotation=45, cmap='Blues')
plt.title('CNN, test set')
plt.show()

How to read the confusion matrix:
- diagonal: correct predictions
- outside the diagonal: errors. Most errors are between **sitting and standing**. For a phone on the waist, these two postures are very similar.

The test accuracy is lower than the validation accuracy. The validation set has only 4 subjects, and these 4 subjects were easier to classify than the 9 test subjects. With few subjects, one estimate can be far from the true value (lecture 04.04).

In [ ]:
report_df = pd.DataFrame(classification_report(y_true, y_pred, target_names=class_names, output_dict=True)).T
report_df.round(3)

## 7. CNN or features?

In Assignment 8, logistic regression with the 561 features of the authors had a test accuracy of about 0.95, which is **higher** than this CNN. The 561 features contain expert knowledge (for example, the angle between the phone and gravity), and 21 subjects are not much data for a model with 200,000 parameters.

A CNN does not need expert features. This is an advantage when nobody knows which features are important, or when there is a lot of data. In Assignment 9, you will compare the two approaches.

**Summary**
- Input of a 1D CNN: (windows, channels, length).
- Scale with the training data only, and keep the validation and test subjects apart.
- Choose the epoch with the validation set. Use the test set one time, at the end.
- A CNN is not always better than a simple model with good features.